# 第02章　逼近的智慧：D组整合实验

本 Notebook 对统稿中的四节核心结论做可重复的数值检查。运行环境为 Python 3.12，
核心依赖为 NumPy、SciPy、Matplotlib 与 SymPy。执行结果由重新启动的内核顺序运行后保存。

基线：`main@c1f4d2b45178ae28bdbacfaedfdfdf8cd1a98839`。主稿和补充稿的选择见同章 `整合-正文.md` 与 D组报告。


## 2.1 插值与 Runge 现象

比较等距节点与 Chebyshev 零点上的高次插值误差。

In [1]:
import json
import numpy as np
from numpy.polynomial.chebyshev import chebpts1

def runge(x):
    return 1.0 / (1.0 + 25.0*x*x)

def lagrange_value(nodes, values, points):
    result = np.zeros_like(points)
    for j, (xj, yj) in enumerate(zip(nodes, values)):
        factors = np.ones_like(points)
        for k, xk in enumerate(nodes):
            if k != j:
                factors *= (points - xk) / (xj - xk)
        result += yj * factors
    return result

degree = 12
x_even = np.linspace(-1, 1, degree + 1)
x_cheb = chebpts1(degree + 1)
test = np.linspace(-1, 1, 1001)
err_even = np.max(np.abs(lagrange_value(x_even, runge(x_even), test) - runge(test)))
err_cheb = np.max(np.abs(lagrange_value(x_cheb, runge(x_cheb), test) - runge(test)))
print(f"等距节点最大误差 = {err_even:.6e}")
print(f"Chebyshev 节点最大误差 = {err_cheb:.6e}")
assert err_cheb < err_even

等距节点最大误差 = 3.662997e+00
Chebyshev 节点最大误差 = 6.921571e-02


## 2.2 最小二乘与换基

同一个多项式空间换到 Chebyshev 基后，精确解不变，但设计矩阵条件数明显下降。

In [2]:
x = np.linspace(-1, 1, 200)
y = np.exp(x) + 0.02*np.cos(7*x)
degree = 8
A_monomial = np.vander(x, degree+1, increasing=True)
A_cheb = np.polynomial.chebyshev.chebvander(x, degree)
c_mono, *_ = np.linalg.lstsq(A_monomial, y, rcond=None)
c_cheb, *_ = np.linalg.lstsq(A_cheb, y, rcond=None)
pred_mono = A_monomial @ c_mono
pred_cheb = A_cheb @ c_cheb
print(f"单项式基条件数 = {np.linalg.cond(A_monomial):.3e}")
print(f"Chebyshev 基条件数 = {np.linalg.cond(A_cheb):.3e}")
print(f"两种基预测最大差 = {np.max(np.abs(pred_mono-pred_cheb)):.3e}")

单项式基条件数 = 5.409e+02
Chebyshev 基条件数 = 3.043e+00
两种基预测最大差 = 4.441e-15


## 2.3 数值积分与自动微分

用复合梯形公式验证二阶收敛，并用中心差分展示截断误差。

In [3]:
import math

def composite_trapezoid(n):
    x = np.linspace(0.0, 1.0, n+1)
    y = np.exp(x)
    h = 1.0 / n
    return h * (0.5*y[0] + y[1:-1].sum() + 0.5*y[-1])

exact_integral = math.e - 1.0
integration_errors = []
for n in [8, 16, 32, 64]:
    error = abs(composite_trapezoid(n) - exact_integral)
    integration_errors.append(error)
    print(f"N={n:2d}, 积分误差={error:.6e}")
ratios = np.asarray(integration_errors[:-1]) / np.asarray(integration_errors[1:])
print("N加倍后的误差比 =", ratios)
assert np.all(ratios > 3.9) and np.all(ratios < 4.1)

N= 8, 积分误差=2.236764e-03
N=16, 积分误差=5.593001e-04
N=32, 积分误差=1.398319e-04
N=64, 积分误差=3.495839e-05
N加倍后的误差比 = [3.99921906 3.99980471 3.99995117]


## 2.4 网络逼近与求导

使用单隐层 tanh 网络检查函数值、输入导数和参数导数；这里聚焦计算图与链式法则，
不做训练收敛性结论。

In [4]:
rng = np.random.default_rng(8)
x0 = 0.37
W = rng.normal(0, 0.8, 10)
b = rng.normal(0, 0.2, 10)
a = rng.normal(0, 0.4, 10)

def network(x):
    z = W*x + b
    h = np.tanh(z)
    return np.dot(a, h)

def network_input_derivative(x):
    z = W*x + b
    h = np.tanh(z)
    return np.dot(a, W*(1.0-h*h))

def network_parameter_derivative(x):
    z = W*x + b
    h = np.tanh(z)
    return np.concatenate([h, a*(1.0-h*h)*x, a*(1.0-h*h)])

parameter_vector = np.concatenate([a, W, b])
def value_from_parameters(vector):
    aa, WW, bb = vector[:10], vector[10:20], vector[20:]
    return np.dot(aa, np.tanh(WW*x0 + bb))

analytic = network_parameter_derivative(x0)
numeric = np.empty_like(parameter_vector)
delta = 1e-6
for index in range(parameter_vector.size):
    plus = parameter_vector.copy(); plus[index] += delta
    minus = parameter_vector.copy(); minus[index] -= delta
    numeric[index] = (value_from_parameters(plus)-value_from_parameters(minus))/(2*delta)
relative_error = np.linalg.norm(analytic-numeric)/np.linalg.norm(analytic)
print(f"网络函数值 f(x0) = {network(x0):.8f}")
print(f"输入解析导数 = {network_input_derivative(x0):.8f}")
print(f"参数导数相对误差 = {relative_error:.3e}")
assert relative_error < 1e-7

网络函数值 f(x0) = 0.30013630
输入解析导数 = 0.58379871
参数导数相对误差 = 3.876e-11


## 执行记录

以下单元把关键结果写入同章 `数据/整合-实验摘要.json`，供网页、报告和后续复核共用。

In [5]:
summary = {
    "runge_error_even": float(err_even),
    "runge_error_chebyshev": float(err_cheb),
    "condition_monomial": float(np.linalg.cond(A_monomial)),
    "condition_chebyshev": float(np.linalg.cond(A_cheb)),
    "integration_error_n64": float(integration_errors[-1]),
    "parameter_gradient_relative_error": float(relative_error),
}
data_dir = __import__("pathlib").Path("数据")
data_dir.mkdir(exist_ok=True)
(data_dir/"整合-实验摘要.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2)+"\n",
    encoding="utf-8",
)
print("整合实验摘要已保存到 数据/整合-实验摘要.json")

整合实验摘要已保存到 数据/整合-实验摘要.json


## 结论与边界

本 Notebook 验证的是统稿中的核心数值结论和可复现性；它不替代定理证明、A 组人工判断
或 B/C 的独立复核。图表与结果均来自本次重启内核后的顺序执行，没有沿用源 Notebook 输出。